In [ ]:
import gymnasium as gym
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt

In [ ]:
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size

        # Hyperparameters
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.01
        self.learning_rate = 0.001

        # Replay buffer
        self.memory = deque(maxlen=2000)

        # --- Networks ---
        self.model = self._build_model()        # main network
        self.target_model = self._build_model() # target network
        self.update_target_model()              # sync at start

    def _build_model(self):
        model = models.Sequential()
        model.add(layers.Dense(24, input_dim=self.state_size, activation='relu'))
        model.add(layers.Dense(24, activation='relu'))
        model.add(layers.Dense(self.action_size, activation='linear'))
        model.compile(loss='mse', optimizer=optimizers.Adam(learning_rate=self.learning_rate))
        return model

    def update_target_model(self):
        """Copy weights from main model to target model"""
        self.target_model.set_weights(self.model.get_weights())

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        q_values = self.model.predict(state, verbose=0)
        return np.argmax(q_values[0])

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return

        minibatch = random.sample(self.memory, batch_size)
        for state, action, reward, next_state, done in minibatch:
            # ----- Double DQN target -----
            if done:
                target = reward
            else:
                best_action = np.argmax(self.model.predict(next_state, verbose=0)[0])
                target_q = self.target_model.predict(next_state, verbose=0)[0][best_action]
                target = reward + self.gamma * target_q

            target_f = self.model.predict(state, verbose=0)
            target_f[0][action] = target
            self.model.fit(state, target_f, epochs=1, verbose=0)

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay


In [ ]:
# Create the CartPole environment
# The 'v1' indicates the version of the environment
env = gym.make('CartPole-v1')

# Get state and action space dimensions
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

# Instantiate the DQNAgent
agent = DQNAgent(state_size, action_size)

episodes = 500
batch_size = 32
scores = []

for e in range(episodes):
    state, _ = env.reset()
    state = np.reshape(state, [1, state_size])
    done = False
    score = 0

    while not done:
        action = agent.act(state)
        next_state, reward, done, _, _ = env.step(action)
        next_state = np.reshape(next_state, [1, state_size])
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        score += reward
        agent.replay(batch_size)

    # --- Update target network every 10 episodes ---
    if e % 10 == 0:
        agent.update_target_model()

    scores.append(score)
    print(f"Episode {e}/{episodes}, Score: {score}, Epsilon: {agent.epsilon:.2f}")

    if len(scores) > 100 and np.mean(scores[-100:]) > 450:
        print(f"Solved after {e + 1} episodes!")
        break


# Plot the scores over episodes to visualize learning progress
plt.plot(scores)
plt.title("Scores over Episodes")
plt.xlabel("Episode")
plt.ylabel("Score")
plt.show()

print("\nTraining completed.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Episode: 1/500, Score: 15.0, Epsilon: 1.00
Episode: 2/500, Score: 14.0, Epsilon: 1.00
Episode: 3/500, Score: 24.0, Epsilon: 0.90
Episode: 4/500, Score: 16.0, Epsilon: 0.83
Episode: 5/500, Score: 13.0, Epsilon: 0.77
Episode: 6/500, Score: 9.0, Epsilon: 0.74
Episode: 7/500, Score: 13.0, Epsilon: 0.69
Episode: 8/500, Score: 22.0, Epsilon: 0.62
Episode: 9/500, Score: 21.0, Epsilon: 0.56
Episode: 10/500, Score: 33.0, Epsilon: 0.47
Episode: 11/500, Score: 34.0, Epsilon: 0.40
Episode: 12/500, Score: 38.0, Epsilon: 0.33
Episode: 13/500, Score: 38.0, Epsilon: 0.27
Episode: 14/500, Score: 29.0, Epsilon: 0.24
Episode: 15/500, Score: 58.0, Epsilon: 0.18
Episode: 16/500, Score: 74.0, Epsilon: 0.12
Episode: 17/500, Score: 122.0, Epsilon: 0.07
Episode: 18/500, Score: 58.0, Epsilon: 0.05
Episode: 19/500, Score: 81.0, Epsilon: 0.03
Episode: 20/500, Score: 59.0, Epsilon: 0.02
Episode: 21/500, Score: 70.0, Epsilon: 0.02
Episode: 22/500, Score: 124.0, Epsilon: 0.01
Episode: 23/500, Score: 150.0, Epsilon: 

In [ ]:
# Create a new environment with rendering
env_render = gym.make('CartPole-v1', render_mode='human')

# Let the agent play for a few episodes to see its performance
for i in range(5):
    state, _ = env_render.reset()
    state = np.reshape(state, [1, state_size])
    done = False
    score = 0
    while not done:
        # Agent acts using its learned policy (no exploration)
        action = np.argmax(agent.model.predict(state, verbose=0)[0])
        next_state, reward, done, _, _ = env_render.step(action)
        next_state = np.reshape(next_state, [1, state_size])
        state = next_state
        score += reward
    print(f"Test Episode {i+1}: Score = {score}")

env_render.close()
print("Rendering finished.")